# Linear Regression & Ridge Regression Fundamentals

A focused end-to-end regression exercise showing ordinary least-squares **Linear Regression** explicitly, then comparing it with a naive median baseline and Ridge regularisation. The notebook keeps the real implementation visible: data creation, missing-value handling, categorical encoding, scaling, train/test separation, MAE/RMSE/R² evaluation, residual diagnostics and example inference.


## 1. Problem
Estimate house prices from size, bedroom count, property age and area. The goal is not only to fit a line: it is to demonstrate a leakage-safe regression workflow and compare Linear Regression against sensible alternatives.


In [ ]:
from __future__ import annotations
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TARGET = 'price'
NUMERIC_FEATURES = ['size_m2', 'bedrooms', 'property_age']
CATEGORICAL_FEATURES = ['area']


## 2. Build and inspect the dataset


In [ ]:
def build_data(n: int = 320) -> pd.DataFrame:
    rng = np.random.default_rng(RANDOM_STATE)
    size = rng.normal(82, 19, n).clip(35, 165)
    bedrooms = rng.integers(1, 6, n)
    property_age = rng.integers(0, 70, n)
    area = rng.choice(['north', 'south', 'east', 'west'], n, p=[0.25] * 4)
    area_effect = np.select(
        [area == 'south', area == 'west', area == 'north'],
        [28000, 16000, 8000],
        default=0,
    )
    price = (
        65000
        + size * 2850
        + bedrooms * 16500
        - property_age * 850
        + area_effect
        + rng.normal(0, 17500, n)
    )
    frame = pd.DataFrame({
        'size_m2': size,
        'bedrooms': bedrooms,
        'property_age': property_age,
        'area': area,
        TARGET: price,
    })
    frame.loc[[5, 19, 77, 201], 'size_m2'] = np.nan
    frame.loc[[11, 99], 'property_age'] = np.nan
    return frame

df = build_data()
print(df.head())
print('shape:', df.shape)
print('missing values:', df.isna().sum().to_dict())
print(df.describe(include='all'))


## 3. Leakage-safe preprocessing
Numerical missing values are imputed from the training split, numerical columns are standardised, and the categorical area feature is one-hot encoded inside a `ColumnTransformer`.


In [ ]:
def build_preprocessor() -> ColumnTransformer:
    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
    ])
    return ColumnTransformer([
        ('numeric', numeric_pipeline, NUMERIC_FEATURES),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_FEATURES),
    ])


## 4. Baseline, Linear Regression and Ridge
The ordinary `LinearRegression()` model is the core algorithm demonstrated here. Ridge is included to show how L2 regularisation changes the modelling choice while keeping the same preprocessing pipeline.


In [ ]:
X = df.drop(columns=TARGET)
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE
)

models = {
    'median_baseline': DummyRegressor(strategy='median'),
    'linear_regression': Pipeline([
        ('prep', build_preprocessor()),
        ('model', LinearRegression()),
    ]),
    'ridge_regression': Pipeline([
        ('prep', build_preprocessor()),
        ('model', Ridge(alpha=1.0)),
    ]),
}

def evaluate(y_true, predictions):
    return {
        'mae': mean_absolute_error(y_true, predictions),
        'rmse': mean_squared_error(y_true, predictions) ** 0.5,
        'r2': r2_score(y_true, predictions),
    }

rows = []
fitted = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append({'model': name, **evaluate(y_test, pred)})
    fitted[name] = model

results = pd.DataFrame(rows).sort_values('mae').reset_index(drop=True)
results


## 5. Residual diagnostics and interpretation


In [ ]:
linear_pred = fitted['linear_regression'].predict(X_test)
residuals = np.asarray(y_test) - np.asarray(linear_pred)
diagnostics = {
    'mean_residual': residuals.mean(),
    'median_abs_residual': np.median(np.abs(residuals)),
    'p90_abs_residual': np.quantile(np.abs(residuals), 0.90),
}
print({k: round(float(v), 2) for k, v in diagnostics.items()})

comparison = pd.DataFrame({
    'actual': np.asarray(y_test),
    'predicted': linear_pred,
    'residual': residuals,
})
comparison['abs_error'] = comparison['residual'].abs()
comparison.sort_values('abs_error', ascending=False).head(10)


## 6. Inference example


In [ ]:
example = pd.DataFrame([{
    'size_m2': 92.0,
    'bedrooms': 3,
    'property_age': 12,
    'area': 'south',
}])
example_prediction = fitted['linear_regression'].predict(example)[0]
print(f'Linear Regression example prediction: £{example_prediction:,.0f}')


## 7. What this proves
This focused notebook demonstrates ordinary least-squares Linear Regression directly, not only a regularised variant. It also shows the surrounding workflow recruiters expect: baseline comparison, preprocessing pipelines, holdout evaluation, MAE/RMSE/R², residual analysis and inference. Larger portfolio projects such as UK House Price Prediction and Statistical Marketing Mix apply regression at greater depth on real data.
